In [2]:
import yfinance as yf
import pandas as pd

In [3]:
dat = yf.Ticker("AAPL")   

# Daily prices (adjusted), then take last trading day of each month
d = dat.history(period="17y", interval="1d", auto_adjust=True)

s = d["Close"].tz_localize(None).resample("M").last()   # Series indexed by month-end
s.index.name = "month_end"

# Make month_end a normal column and add an integer index column
monthly = s.reset_index(name="adj_close")
monthly.insert(0, "idx", range(1, len(monthly)+1))   # 1-based; use range(len(monthly)) for 0-based

#Change column names
monthly=monthly.rename(columns={"month_end":"Date", "adj_close":"Price"})


#Drop idx column
monthly=monthly.drop(columns=["idx"])

#Convert to DataFrame
microsoft_price = pd.DataFrame(monthly)

# Add column called calendar_quarter

p = microsoft_price["Date"].dt.to_period("Q")        # calendar quarters
microsoft_price["cal_year"] = p.dt.year.astype("Int64")    # 2010, 2011, ...
microsoft_price["cal_q"]    = p.dt.quarter                 # 1..4

# Final label: "Q{cal_q} {cal_year}"
microsoft_price["calendar_quarter"] = "Q" + microsoft_price["cal_q"].astype(str) + " " + microsoft_price["cal_year"].astype(str)


#Drop cal_year and cal_q column
microsoft_price=microsoft_price.drop(columns=["cal_year", "cal_q"])


microsoft_price


/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_10140/3729836985.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  s = d["Close"].tz_localize(None).resample("M").last()   # Series indexed by month-end


,Date,Price,calendar_quarter
0,2008-09-30,3.412087,Q3 2008
1,2008-10-31,3.229864,Q4 2008
2,2008-11-30,2.781965,Q4 2008
3,2008-12-31,2.562217,Q4 2008
4,2009-01-31,2.705713,Q1 2009
...,...,...,...
200,2025-05-31,200.622314,Q2 2025
201,2025-06-30,204.937408,Q2 2025
202,2025-07-31,207.334702,Q3 2025
203,2025-08-31,232.139999,Q3 2025


In [4]:
microsoft_price.to_csv("apple_price.csv")